# Task 7: BERT Pretraining

Implementing MLM (Masked Language Modeling) and NSP (Next Sentence Prediction).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)

## Masked Language Modeling (MLM)

Randomly mask 15% of tokens and predict them:
- 80% replaced with [MASK]
- 10% replaced with random token
- 10% kept as original

In [ ]:
def create_mlm_labels(token_ids, mask_token_id=103, vocab_size=30000, mask_prob=0.15):
    """Create MLM training data with masking."""
    labels = token_ids.clone()
    probability_matrix = torch.full(token_ids.shape, mask_prob)
    
    # Don't mask special tokens
    special_tokens_mask = (token_ids == 0) | (token_ids == 101) | (token_ids == 102)  # [PAD], [CLS], [SEP]
    probability_matrix.masked_fill_(special_tokens_mask, value=0.0)
    
    # Sample masked positions
    masked_indices = torch.bernoulli(probability_matrix).bool()
    labels[~masked_indices] = -100  # Only compute loss on masked tokens
    
    # 80% [MASK], 10% random, 10% unchanged
    indices_replaced = (torch.bernoulli(torch.full(token_ids.shape, 0.8))).bool() & masked_indices
    token_ids[indices_replaced] = mask_token_id
    
    indices_random = (torch.bernoulli(torch.full(token_ids.shape, 0.5))).bool() & masked_indices & ~indices_replaced
    random_tokens = torch.randint(4, vocab_size - 1, token_ids.shape)
    token_ids[indices_random] = random_tokens[indices_random]
    
    return token_ids, labels

# Test
token_ids = torch.tensor([[101, 2000, 3000, 4000, 102, 0, 0]])
masked_ids, labels = create_mlm_labels(token_ids)
print(f"Original: {token_ids.tolist()}")
print(f"Masked:   {masked_ids.tolist()}")
print(f"Labels:   {labels.tolist()}  (-100 = ignore)")

## Next Sentence Prediction (NSP)

Predict if sentence B follows sentence A (IsNext or NotNext).

In [ ]:
class NSPHead(nn.Module):
    """Next Sentence Prediction head."""
    
    def __init__(self, d_model):
        super().__init__()
        self.linear = nn.Linear(d_model, 2)  # IsNext, NotNext
        
    def forward(self, pooled_output):
        return self.linear(pooled_output)

class MLMHead(nn.Module):
    """Masked Language Modeling head (BERT uses it to predict masked tokens)."""
    
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.dense = nn.Linear(d_model, d_model)
        self.activation = nn.GELU()
        self.layer_norm = nn.LayerNorm(d_model)
        self.decoder = nn.Linear(d_model, vocab_size)
        
    def forward(self, x):
        x = self.dense(x)
        x = self.activation(x)
        x = self.layer_norm(x)
        x = self.decoder(x)
        return x

# Test
mlm_head = MLMHead(d_model=256, vocab_size=30000)
nsp_head = NSPHead(d_model=256)

sequence_output = torch.randn(2, 128, 256)  # All token outputs
pooled_output = torch.randn(2, 256)  # [CLS] output

mlm_logits = mlm_head(sequence_output)
nsp_logits = nsp_head(pooled_output)

print(f"MLM logits: {mlm_logits.shape}")
print(f"NSP logits: {nsp_logits.shape}")

In [ ]:
class BERTPretrained(nn.Module):
    """Complete BERT model for pretraining."""
    
    def __init__(self, vocab_size, d_model=768, num_heads=12, num_layers=12, 
                 d_ff=3072, max_len=512, dropout=0.1):
        super().__init__()
        
        # Encoder
        self.encoder = BERTEncoder(vocab_size, d_model, num_heads, num_layers, d_ff, max_len, dropout)
        
        # MLM and NSP heads
        self.mlm_head = MLMHead(d_model, vocab_size)
        self.nsp_head = NSPHead(d_model)
        
    def forward(self, token_ids, segment_ids=None, attention_mask=None, mlm_labels=None):
        # Get encoder outputs
        sequence_output, pooled_output = self.encoder(token_ids, segment_ids, attention_mask)
        
        # MLM predictions
        mlm_logits = self.mlm_head(sequence_output)
        
        # NSP predictions  
        nsp_logits = self.nsp_head(pooled_output)
        
        # Compute MLM loss if labels provided
        loss = None
        if mlm_labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            mlm_loss = loss_fct(mlm_logits.view(-1, mlm_logits.size(-1)), mlm_labels.view(-1))
            
            nsp_loss = F.cross_entropy(nsp_logits, torch.randint(0, 2, (token_ids.size(0),)))
            loss = mlm_loss + nsp_loss
        
        return loss, mlm_logits, nsp_logits


class BERTEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=768, num_heads=12, num_layers=12, 
                 d_ff=3072, max_len=512, dropout=0.1):
        super().__init__()
        self.embeddings = BERTEmbeddings(vocab_size, d_model, max_len)
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.pooler = nn.Linear(d_model, d_model)
        
    def forward(self, token_ids, segment_ids=None, attention_mask=None):
        x = self.embeddings(token_ids, segment_ids)
        for layer in self.encoder_layers:
            x = layer(x)
        pooled = torch.tanh(self.pooler(x[:, 0]))
        return x, pooled


class BERTEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model, max_len=512):
        super().__init__()
        self.token = nn.Embedding(vocab_size, d_model)
        self.segment = nn.Embedding(2, d_model)
        self.position = nn.Embedding(max_len, d_model)
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, token_ids, segment_ids=None):
        seq_len = token_ids.size(1)
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0).expand_as(token_ids)
        segment = segment_ids if segment_ids is not None else torch.zeros_like(token_ids)
        return self.norm(self.token(token_ids) + self.segment(segment) + self.position(positions))


class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(nn.Linear(embed_dim, d_ff), nn.GELU(), nn.Dropout(dropout), 
                                nn.Linear(d_ff, embed_dim), nn.Dropout(dropout))
        
    def forward(self, x, mask=None):
        x = self.norm1(x + self.attn(x, x, x)[0])
        x = self.norm2(x + self.ffn(x))
        return x

# Demo with small model
model = BERTPretrained(vocab_size=30000, d_model=128, num_heads=4, num_layers=2, d_ff=512)
token_ids = torch.randint(0, 30000, (2, 64))

loss, mlm_logits, nsp_logits = model(token_ids)
print(f"Loss: {loss.item():.4f}")
print(f"MLM logits: {mlm_logits.shape}")
print(f"NSP logits: {nsp_logits.shape}")

## Summary
- ✅ MLM: Mask 15%, predict masked tokens
- ✅ NSP: Binary classification (IsNext/NotNext)
- ✅ Combined loss for pretraining

## Next: [Task 8: BERT Finetuning - Classification](../task-08-bert-classification/overview.html)